# Export the clothing model (Colab)

CI does this automatically on every push that changes `tools/categories.yaml` or `tools/export_models.py`. Use this notebook only to experiment, e.g. to try new prompts and see the outputs.

Runtime → Run all. The files appear in the Colab file browser under `gallery-organizer/android/app/src/main/assets/models/clothing/`.

In [ ]:
!git clone --depth 1 https://github.com/SyedUsmanAkhlaq/gallery-organizer.git
%cd gallery-organizer
!pip install -q -r tools/requirements.txt

In [ ]:
# Optional: edit tools/categories.yaml in the file browser first.
!python tools/export_models.py

In [ ]:
# Quick check: classify an image of your choice (upload it via the file browser).
import json, numpy as np, onnxruntime as ort
from PIL import Image
d = 'android/app/src/main/assets/models/clothing/'
meta = json.load(open(d + 'text_embeddings.json'))
sess = ort.InferenceSession(d + 'image_encoder.onnx')
def classify(path):
    s = meta['image_size'][0]
    im = Image.open(path).convert('RGB')
    w, h = im.size; side = min(w, h)
    im = im.crop(((w-side)//2, (h-side)//2, (w-side)//2+side, (h-side)//2+side)).resize((s, s), Image.BICUBIC)
    x = (np.asarray(im, np.float32)/255 - meta['mean']) / meta['std']
    e = sess.run(None, {'pixels': x.transpose(2,0,1)[None].astype(np.float32)})[0][0]
    T = np.array([c['embedding'] for c in meta['categories']])
    z = meta['logit_scale'] * T @ e; p = np.exp(z - z.max()); p /= p.sum()
    for i in p.argsort()[::-1][:3]: print(f"{meta['categories'][i]['label']:16s} {p[i]:.2f}")
# classify('my_photo.jpg')